In [ ]:
# ============================================================
# 0. Setup: libraries and plotting defaults
# ============================================================
# Core data handling
import sys
import numpy as np
import pandas as pd
import os

# Plotting: %matplotlib inline renders figures directly below each cell;
# the mpl.rc calls bump default font sizes so axis labels/ticks stay
# readable when figures are shown at notebook width
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

import tarfile
import urllib

# 1. Get the Data

Load the raw marketing campaign survey (`marketing_campaign.csv`) into a `DataFrame`. This is the only step that touches disk — everything downstream works off `df_marketing`.

In [ ]:
# The raw file is ";"-separated, so make that explicit rather than
# relying on pandas' default comma delimiter
marketing_campaign_data_path = os.path.join("datasets", "marketing_campaign")

def get_data(path_data=marketing_campaign_data_path):
    csv_file = os.path.join(path_data, "marketing_campaign.csv")
    return pd.read_csv(csv_file, sep=";")

df_marketing = get_data()
df_marketing.head()

# 2. Data Understanding

Explore the raw data before touching it: column types, summary statistics, the target class balance, and how customers behave (spending, channels, campaign responses). The goal here is to spot cleaning needs and candidate features, not to fix anything yet.

In [ ]:
# Data types & column counts: confirms which columns are numeric vs.
# categorical before we decide how to encode/clean each one
df_marketing.info()

num_cols = df_marketing.select_dtypes(include=['number'])
cat_cols = df_marketing.select_dtypes(include=['object', 'category'])

print("Numerical columns in the data set:", num_cols.shape[1])
print("Categorical columns in the data set:", cat_cols.shape[1])
print("Number of columns in the data set", (num_cols + cat_cols).shape[1])

In [ ]:
# Summary statistics for numerical columns - useful to spot outliers
# (e.g. Income max of 666,666 vs. a mean of ~52k) and skewed ranges
df_marketing.describe()

In [ ]:
# Target variable: check how imbalanced "Response" is, since that
# drives choices later on (class_weight, threshold tuning, etc.)
print("Counts of response variable")
print(df_marketing["Response"].value_counts())

In [ ]:
import seaborn as sns
import math

# ------------------------------------------------------------------
# Palette: a small, fixed set of colors so the same hue always means
# the same thing across every plot below (blue = primary series,
# green/gray reserved for "accepted" vs "not accepted")
# ------------------------------------------------------------------
COLOR_ACCEPTED = "#0ca30c"      # accepted the campaign offer
COLOR_NOT_ACCEPTED = "#898781"  # did not accept
CAT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
SEQ_BLUE = "#2a78d6"

# ------------------------------------------------------------------
# Campaign behavior: which of the 5 past campaigns got the most
# "yes" responses?
# ------------------------------------------------------------------
print("Campaign behavior".center(50, "-"))

camp_response_list = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']
campaign_counts = df_marketing[camp_response_list].sum()
most_accepted_campaign = campaign_counts.idxmax()
print("The most accepted campaign is: ", most_accepted_campaign[8:])

# reshape to long format: one row per (campaign, accepted/not) so
# seaborn can group bars by campaign and color by response
df_reshape = df_marketing[camp_response_list].melt(var_name="Campaign", value_name="Accepted")

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(
    data=df_reshape, x="Campaign", hue="Accepted",
    palette=[COLOR_NOT_ACCEPTED, COLOR_ACCEPTED], ax=ax
)
ax.set_xticklabels([c[8:] for c in camp_response_list])  # drop the "AcceptedCmp" prefix
ax.set_xlabel("Campaign")
ax.set_ylabel("Number of Customers")
ax.set_title("Customer Responses per Campaign")
ax.legend(title="Response", labels=["Not accepted", "Accepted"], frameon=False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
plt.close()

# ------------------------------------------------------------------
# Personal info: income distribution
# ------------------------------------------------------------------
print("Customer personal information".center(50, "-"))

mean_in = df_marketing["Income"].mean()
median_in = df_marketing["Income"].median()
print(f"The mean income value is: {mean_in:,.2f} USD")
print(f"The median income value is: {median_in:,.2f} USD")

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(df_marketing["Income"], bins=50, range=(0, 200000), color=SEQ_BLUE, edgecolor="white", label="Customers")
ax.axvline(mean_in, color="#e34948", linestyle="--", linewidth=2, label=f"Mean (${mean_in:,.0f})")
ax.axvline(median_in, color="#4a3aa7", linestyle=":", linewidth=2, label=f"Median (${median_in:,.0f})")
ax.set_xlabel("Income (USD)")
ax.set_ylabel("Number of Customers")
ax.set_title("Customer Yearly Household Income")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
plt.close()

# ------------------------------------------------------------------
# Household composition: marital status, kids, teenagers at home
# ------------------------------------------------------------------
fig, axs = plt.subplots(1, 3, figsize=(13, 5))
axs = axs.flatten()

print("The most common marital status is:", df_marketing["Marital_Status"].value_counts().idxmax())
print("The most frequent number of small children is:", df_marketing["Kidhome"].value_counts().idxmax())
print("The most frequent number of teenagers is:", df_marketing["Teenhome"].value_counts().idxmax())

household_fields = [
    ("Marital_Status", "Marital Status", 0),
    ("Kidhome", "Kids at Home", 1),
    ("Teenhome", "Teenagers at Home", 2),
]

for i, (col, title, slot) in enumerate(household_fields):
    counts = df_marketing[col].value_counts()
    axs[i].bar(counts.index.astype(str), counts.values, color=CAT_PALETTE[i])
    axs[i].set_title(title)
    axs[i].set_ylabel("Number of Customers")
    axs[i].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
plt.close()

# ------------------------------------------------------------------
# Spending behavior: how much customers spend per product category
# ------------------------------------------------------------------
print("How much the customer spends in the products".center(50, "-"))

amount_prod_list = ['MntWines', 'MntMeatProducts', 'MntGoldProds', 'MntFishProducts', 'MntSweetProducts', 'MntFruits']
product_totals = df_marketing[amount_prod_list].sum().sort_values(ascending=False)
print("Product that customer spends more on is:", product_totals.idxmax()[3:])

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    [c[3:] for c in product_totals.index], product_totals.values,
    color=CAT_PALETTE[:len(product_totals)]
)
ax.set_ylabel("Total Amount Spent (USD)")
ax.set_title("Total Customer Spending by Product Category")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()
plt.close()

df_marketing[amount_prod_list].describe()

# ------------------------------------------------------------------
# Purchase channel behavior: where do customers buy?
# ------------------------------------------------------------------
print("Customer interactions with company".center(50, "-"))

purchase_list = ['NumStorePurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumDealsPurchases']
purchase_totals = df_marketing[purchase_list].sum().sort_values(ascending=False)
print("The most common type of purchase is:", purchase_totals.idxmax()[3:])

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(
    [c[3:] for c in purchase_totals.index], purchase_totals.values,
    color=CAT_PALETTE[:len(purchase_totals)]
)
ax.set_ylabel("Total Number of Purchases")
ax.set_title("Purchases by Channel")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()
plt.close()

# ------------------------------------------------------------------
# Recency: how many days since each customer's last purchase
# ------------------------------------------------------------------
print("Days since the last purchase".center(50, "-"))

print("The most common recency value is:", df_marketing["Recency"].value_counts().idxmax(), "days")

fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(data=df_marketing, x="Recency", bins=100, color=SEQ_BLUE, ax=ax)
ax.set_xlabel("Days Since Last Purchase")
ax.set_ylabel("Number of Customers")
ax.set_title("Recency of Last Purchase")
plt.tight_layout()
plt.show()
plt.close()

# 3. Feature Engineering

Turn raw survey columns into signals that correlate better with the target. We check the correlation of each numeric column against `Response`, then engineer `TotalMntProd` (total 2-year spend) and `TotalPurchase` (total purchase count) to see if aggregating the spending / purchase columns strengthens the relationship. Columns that are constant across the whole dataset (`Z_CostContact`, `Z_Revenue`) are dropped since they carry no information.

In [ ]:
# Correlation of every numeric column with the target "Response":
# ranks candidate predictors before we engineer new features
corr_matrix = df_marketing.select_dtypes(include="number").corr()
corr_matrix["Response"].sort_values(ascending=False)

In [ ]:
# Aggregate the 6 product-spend columns into one "total spend" feature
df_marketing["TotalMntProd"] = (
    df_marketing["MntWines"] + df_marketing["MntFruits"] + df_marketing["MntMeatProducts"]
    + df_marketing["MntFishProducts"] + df_marketing["MntSweetProducts"] + df_marketing["MntGoldProds"]
)
# Aggregate the 3 purchase-channel columns into one "total purchases" feature
df_marketing["TotalPurchase"] = (
    df_marketing["NumWebPurchases"] + df_marketing["NumCatalogPurchases"] + df_marketing["NumStorePurchases"]
)

# Recompute correlations to confirm the engineered features are at
# least as informative as the columns they summarize
corr_matrix = df_marketing.select_dtypes(include="number").corr()
corr_matrix["Response"].sort_values(ascending=False)

In [ ]:
# Z_CostContact and Z_Revenue are constant for every customer (see the
# NaN correlation above), so they carry no predictive signal - drop them
df_marketing = df_marketing.drop(columns=["Z_CostContact", "Z_Revenue"])

## 4. Data Cleaning

Handle missing values and duplicate rows before encoding categorical features or training anything.

In [ ]:
# Check for missing values column by column
print("missing data")
df_marketing.isnull().sum()

In [ ]:
# "Income" is the only column with missing values (24 rows) - impute
# with the median since income is right-skewed (see the histogram
# above) and the median is more robust to the extreme outlier income
median = df_marketing["Income"].median()
df_marketing["Income"] = df_marketing["Income"].fillna(median)
df_marketing["Income"].isnull().sum()

In [ ]:
# Check for fully duplicated rows
print("duplicated values")
df_marketing.duplicated().sum()

## 5. Handling Text and Categorical Attributes

One-hot encode "Education" (an unordered category) and drop "Marital_Status" and "Dt_Customer", which don't add predictive value here and would otherwise need their own encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# "Education" has no natural ordering, so one-hot encode it rather
# than using an ordinal/label encoding
cat_col_name = ["Education"]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(df_marketing[cat_col_name])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(cat_col_name),
    index=df_marketing.index
)

df_marketing = pd.concat(
    [df_marketing.drop(columns=cat_col_name), encoded_df],
    axis=1
)

# Marital_Status and Dt_Customer aren't used as model features here
df_marketing = df_marketing.drop(columns=["Marital_Status", "Dt_Customer"])

In [ ]:
# Keep a dedicated copy for training so any further experimentation
# doesn't accidentally mutate the cleaned df_marketing
df_marketing_prep = df_marketing.copy()
df_marketing_prep.head()

# 6. Train the Model

Split into train/test, fit a `RandomForestClassifier` (with `class_weight="balanced"` to compensate for the ~13% positive-class imbalance seen earlier), then use cross-validated probabilities on the training set to pick a probability threshold that maximizes F1 before scoring the held-out test set.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import precision_recall_curve

# ------------------------------------------------------------------
# Train / test split
# ------------------------------------------------------------------
Y = df_marketing_prep['Response']
X = df_marketing_prep.drop(['Response'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# class_weight="balanced" compensates for the ~13/87 class imbalance
# found in the Data Understanding section
model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

# ------------------------------------------------------------------
# Pick a decision threshold using cross-validated probabilities on
# the training set only, so the test set stays untouched until the
# final evaluation below
# ------------------------------------------------------------------
y_scores = cross_val_predict(model, X_train, y_train, cv=3, method="predict_proba")[:, 1]
precision, recall, thresholds = precision_recall_curve(y_train, y_scores)

def plot_precision_recall_vs_threshold(precisions, recalls, thresholds):
    plt.figure(figsize=(7, 5))
    plt.plot(thresholds, precisions[:-1], color="#2a78d6", linestyle="--", linewidth=2, label="Precision")
    plt.plot(thresholds, recalls[:-1], color="#eb6834", linestyle="-", linewidth=2, label="Recall")
    plt.xlabel("Decision Threshold")
    plt.ylabel("Score")
    plt.title("Precision & Recall vs. Decision Threshold")
    plt.grid(alpha=0.3)
    plt.legend(frameon=False)

plot_precision_recall_vs_threshold(precision, recall, thresholds)

# The threshold that maximizes F1 balances precision and recall
# rather than defaulting to the usual 0.5 cutoff
f1 = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1])
best_threshold_id = np.argmax(f1)
best_threshold = thresholds[best_threshold_id]

plt.axvline(best_threshold, color="#4a3aa7", linestyle=":", linewidth=2, label=f"Best threshold ({best_threshold:.2f})")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

print("Best threshold:", best_threshold)

# ------------------------------------------------------------------
# Final evaluation on the held-out test set using the tuned threshold
# ------------------------------------------------------------------
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > best_threshold).astype(int)

cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(cm)
print(classification_report(y_test, y_pred))

### How to interpret these metrics?

**Precision** — of all customers we *targeted* as likely buyers, what fraction actually accepted the offer? (correctly targeted customers / all targeted customers). We measured an optimal precision of ~49%, meaning about half of the customers we'd contact will convert.

**Recall** — of all customers who *would have* accepted the offer, what fraction did we correctly identify? (correctly targeted customers / all true buyers). We measured an optimal recall of ~61%, meaning we catch about 61% of the customers who would say yes.

# 7. ROI Estimation (Profit / Total Cost)

Translate the confusion matrix into a business number: if we only contact the customers the model flags as likely responders, was it worth the cost? `TP` and `FP` are read directly from the test-set confusion matrix computed above — true positives convert and pay back `gain`, everyone contacted (TP + FP) costs `cost` to reach.

In [ ]:
# TP / FP come straight from the confusion matrix computed above, so
# this estimate always matches the model that was actually evaluated
tn, fp, fn, tp = cm.ravel()

gain = 20  # USD earned when a contacted customer accepts the offer
cost = 1   # USD spent to contact one customer

profit = tp * gain - (tp + fp) * cost
roi = profit / ((tp + fp) * cost) * 100

print(f"Customers contacted (TP + FP): {tp + fp}")
print(f"Estimated profit: ${profit:,.2f} USD")
print(f"ROI: {roi:.1f}%")
print(f"For every 1 USD spent, you earn: ${roi / 100:.2f} USD")

# 8. Evaluate the Model on "New Customers"

Score every customer in the held-out test set — standing in for a batch of new customers — and turn the raw probabilities into something a marketing team can act on: a ranked shortlist, outreach-priority tiers, and the expected profit if we contact everyone the model flags.

In [ ]:
# ------------------------------------------------------------------
# Score every held-out customer and rank by predicted probability
# of responding to the offer
# ------------------------------------------------------------------
df_test = X_test.copy()
df_test["prob_response"] = y_prob
df_test["prediction"] = y_pred
df_test = df_test.sort_values("prob_response", ascending=False).reset_index(drop=True)

# ------------------------------------------------------------------
# Bucket customers into outreach-priority tiers so the result reads
# as an action list rather than a wall of probabilities
# ------------------------------------------------------------------
def assign_tier(prob, threshold):
    if prob >= max(threshold, 0.66):
        return "High priority"
    elif prob >= threshold:
        return "Medium priority"
    return "Low priority"

df_test["tier"] = df_test["prob_response"].apply(lambda p: assign_tier(p, best_threshold))

n_contact = int((df_test["prediction"] == 1).sum())
expected_responders = df_test.loc[df_test["prediction"] == 1, "prob_response"].sum()
expected_profit = expected_responders * gain - n_contact * cost

print("New customers scoring summary".center(60, "="))
print(f"Customers scored:              {len(df_test):>6}")
print(f"Flagged to contact (>{best_threshold:.2f}):    {n_contact:>6}")
print(f"Expected responders (est.):    {expected_responders:>6.1f}")
print(f"Expected profit (est.):        ${expected_profit:>8.2f} USD")

print()
print("Top 10 customers by predicted response probability".center(60, "-"))
top10 = df_test[["ID", "prob_response", "tier"]].head(10).copy()
top10["prob_response"] = top10["prob_response"].map(lambda p: f"{p:.1%}")
print(top10.to_string(index=False))

# ------------------------------------------------------------------
# Visualize the size of each outreach tier
# ------------------------------------------------------------------
tier_order = ["High priority", "Medium priority", "Low priority"]
tier_colors = {"High priority": "#0ca30c", "Medium priority": "#eda100", "Low priority": "#898781"}
counts = df_test["tier"].value_counts().reindex(tier_order).fillna(0)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color=[tier_colors[t] for t in tier_order])
for i, v in enumerate(counts.values):
    ax.text(i, v + 0.5, f"{int(v)}", ha="center", va="bottom")
ax.set_ylabel("Number of Customers")
ax.set_title("New Customers by Outreach Priority")
plt.tight_layout()
plt.show()
plt.close()